### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 
    Tracking agent behavior with logging, analytics, and debugging.
    Transforming prompts, tool selection, and output formatting.
    Adding retries, fallbacks, and early termination logic.
    Applying rate limits, guardrails, and Pll detection.


In [12]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
    Long Running conversations that exceed context windows.

    Multi-turn dialogues with extensive history.

    Applications where preserving full conversation context matters.

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [14]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [15]:
# Alternative test data
questions = [
    "What is 2 + 2?",
    "What is 10*5",
    "What is 100/4",
    "What is 15-7?",
    "What is 3*3",
    "What is 4*4",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2 + 2?', additional_kwargs={}, response_metadata={}, id='cd0b15a1-d5b4-4a82-b899-1104d05bea30'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_45299bce03', 'id': 'chatcmpl-EcBqM6mR8hdO2cRfBP2baiXZYdT4E', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16975-e5b8-70f1-ad06-251da0afd5e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'tot

# token Size

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
        model = "gpt-4o-mini",
        tools = [search_hotels],
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger = ("tokens", 550),
                keep=("tokens", 200)
            )
        ]
    )

config = {"configurable": {"thread_id": "test-1"}}

    # Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4    

In [22]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~143 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='663068f4-f89e-4df5-a043-84738153ab13'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d7a8a867ce', 'id': 'chatcmpl-EcC3vA4Ggp2vpEZ8fqnibXN1cG9c2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16982-bf74-7141-b61e-72ad9c28f680-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_bTC1kNJHlu7ouJ

## Fraction

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
        model = "gpt-4o-mini",
        tools = [search_hotels],
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger = ("fraction", 0.005),
                keep=("fraction", 0.002)
            )
        ]
    )

config = {"configurable": {"thread_id": "test-1"}}

    # Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  

In [25]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~146 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='53573e5d-821d-43b9-8b48-1861d6b22f01'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d7a8a867ce', 'id': 'chatcmpl-EcC8Ni7BIEPmKepn8Az3TzogtnGdo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16986-f524-7de0-9c78-50178977b509-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_3c901ifMJmkJtD